# 5.17 Stochastic variational inference

SVI makes VI scalable by replacing full-data updates with noisy, scaled minibatch updates.

SVI updates global variational parameters using minibatch sufficient statistics and a learning-rate schedule. We implement Beta-Bernoulli and Dirichlet-categorical SVI on a D1-D5 ladder.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 517
rng = np.random.default_rng(SEED)


def svi_beta_bernoulli_minibatch(data, N, rho, prior=np.array([1.0, 1.0]), current=np.array([1.0, 1.0])):
    batch = np.asarray(data, dtype=float)
    scale = N / len(batch)
    successes = batch.sum()
    failures = len(batch) - successes
    scaled_update = prior + scale * np.array([successes, failures])
    new_param = (1.0 - rho) * current + rho * scaled_update
    return new_param, scaled_update


def beta_mean(param):
    return param[0] / param.sum()


def svi_beta_path(data, batch_size=5, steps=80, tau=4.0, kappa=0.7, scale_batches=True, constant_rho=None):
    N = len(data)
    lam = np.array([1.0, 1.0])
    path = []
    truth = np.array([1.0 + data.sum(), 1.0 + N - data.sum()])
    for t in range(steps):
        idx = rng.choice(N, size=batch_size, replace=False)
        batch = data[idx]
        rho = constant_rho if constant_rho is not None else (t + tau) ** (-kappa)
        if scale_batches:
            lam_hat = np.array([1.0, 1.0]) + (N / batch_size) * np.array([batch.sum(), batch_size - batch.sum()])
        else:
            lam_hat = np.array([1.0, 1.0]) + np.array([batch.sum(), batch_size - batch.sum()])
        lam = (1.0 - rho) * lam + rho * lam_hat
        error = np.linalg.norm(lam - truth) / np.linalg.norm(truth)
        path.append((lam.copy(), float(error)))
    return lam, np.array([p[1] for p in path]), truth


def build_svi_ladder():
    d1 = np.array([1, 1, 1, 1, 1, 1, 1, 0, 0, 0])
    d2 = rng.choice([0, 1, 2, 3], size=80, p=[0.1, 0.2, 0.3, 0.4])
    d3 = np.r_[rng.binomial(1, 0.15, size=60), rng.binomial(1, 0.85, size=60)]
    table = np.array([[42, 8], [35, 15], [9, 21], [12, 28]])
    d4 = np.r_[np.ones(table[:, 0].sum()), np.zeros(table[:, 1].sum())]
    sparse = rng.binomial(1, 0.04, size=400)
    bursts = rng.choice(len(sparse), size=25, replace=False)
    sparse[bursts] = 1
    return [
        {"name": "D1 Bernoulli exact", "data": d1, "kind": "beta"},
        {"name": "D2 four-state categorical global parameter", "data": d2, "kind": "dirichlet", "K": 4},
        {"name": "D3 noisy bimodal minibatches", "data": d3, "kind": "beta"},
        {"name": "D4 click/conversion table", "data": d4, "kind": "beta"},
        {"name": "D5 sparse minibatches with bad schedule", "data": sparse, "kind": "beta"},
    ]


def full_dirichlet(data, K):
    counts = np.bincount(data, minlength=K)
    return np.ones(K) + counts


def svi_dirichlet_path(data, K, batch_size=10, steps=80):
    N = len(data)
    lam = np.ones(K)
    truth = full_dirichlet(data, K)
    errors = []
    for t in range(steps):
        idx = rng.choice(N, size=batch_size, replace=False)
        batch = data[idx]
        counts = np.bincount(batch, minlength=K)
        lam_hat = np.ones(K) + (N / batch_size) * counts
        rho = (t + 4.0) ** (-0.7)
        lam = (1.0 - rho) * lam + rho * lam_hat
        errors.append(float(np.linalg.norm(lam - truth) / np.linalg.norm(truth)))
    return lam, np.array(errors), truth


## The concept, built once: scaled minibatch natural-parameter updates

SVI uses

$$\lambda_t=(1-\rho_t)\lambda_{t-1}+\rho_t\hat\lambda_t,\quad \rho_t=(t+\tau)^{-\kappa}.$$

A minibatch must be scaled so it represents the full dataset.

In [ ]:

data = np.array([1, 1, 1, 1, 1, 1, 1, 0, 0, 0])
full_update = np.array([1 + 7, 1 + 3])
print("full update", full_update)
assert np.allclose(full_update, [8, 4])

batch = np.array([1, 0, 1, 1])
new_param, scaled = svi_beta_bernoulli_minibatch(batch, N=10, rho=0.3)
print("scaled minibatch", scaled)
assert np.allclose(scaled, [8.5, 3.5])


The learning rate damps the noisy minibatch update into the previous global parameter.

In [ ]:

current = np.array([1.0, 1.0])
rho = 0.3
damped = (1.0 - rho) * current + rho * scaled
print("damped update", damped)
assert np.allclose(damped, [3.25, 1.75])

scale = 10 / 4
assert np.isclose(scale, 2.5)


## The dataset ladder

The ladder grows from one Bernoulli posterior to noisy, tabular, and sparse minibatch regimes.

In [ ]:

ladder = build_svi_ladder()
for i, rung in enumerate(ladder, start=1):
    data = rung["data"]
    print(i, rung["name"])
    print("kind", rung["kind"], "N", len(data))
    print("sample", data[:12])


In [ ]:

rows = []
paths = []
params = []
truths = []
for i, rung in enumerate(ladder, start=1):
    if rung["kind"] == "beta":
        lam, errors, truth = svi_beta_path(rung["data"], batch_size=min(10, len(rung["data"])), steps=90)
    else:
        lam, errors, truth = svi_dirichlet_path(rung["data"], rung["K"], batch_size=12, steps=90)
    rows.append((f"D{i}", rung["name"], float(errors[-1])))
    paths.append(errors)
    params.append(lam)
    truths.append(truth)

print("rung | marginal parameter error")
for label, name, error in rows:
    print(f"{label} | {name} | {error:.6f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(17, 6))
for i, rung in enumerate(ladder):
    ax = axes[0, i]
    lam = params[i]
    truth = truths[i]
    x = np.arange(len(lam))
    ax.bar(x - 0.18, truth / truth.sum(), width=0.36, label="full")
    ax.bar(x + 0.18, lam / lam.sum(), width=0.36, label="SVI")
    ax.set_title(f"D{i + 1}")
    if i == 0:
        ax.legend(fontsize=8)

ax = axes[1, 0]
for i, errors in enumerate(paths):
    ax.plot(errors, label=f"D{i + 1}")
ax.set_title("marginal-error-vs-iteration")
ax.set_xlabel("iteration")
ax.set_ylabel("relative parameter error")
ax.legend(fontsize=8)
for j in range(1, 5):
    axes[1, j].axis("off")
plt.tight_layout()


## Pitfall on D5: missing scaling and constant large steps

Without N/B scaling, minibatches are biased small. With a constant large step, the stochastic path bounces instead of settling.

In [ ]:

d5 = ladder[-1]["data"]
good_lam, good_errors, truth = svi_beta_path(d5, batch_size=20, steps=100, scale_batches=True, constant_rho=None)
small_lam, small_errors, _ = svi_beta_path(d5, batch_size=20, steps=100, scale_batches=False, constant_rho=None)
bouncy_lam, bouncy_errors, _ = svi_beta_path(d5, batch_size=20, steps=100, scale_batches=True, constant_rho=0.8)
print("good final error", good_errors[-1])
print("unscaled final error", small_errors[-1])
print("constant-step tail std", np.std(bouncy_errors[-20:]))
assert good_errors[-1] < small_errors[-1]


## Evaluate it + practice

- Metric: relative marginal parameter error versus full-batch posterior/VI reference.
- No-skill baseline: update with raw minibatch counts as if B observations were the whole dataset.
- Cheap sanity check: the D1 full update is [8, 4] and the scaled batch update is [8.5, 3.5].
- Ablation: remove minibatch scaling or force constant rho and D5 error increases or bounces.
- Failure signals: posterior parameters remain too small, held-out diagnostics are noisy, or path fails to settle.

**Practice.** Try batch sizes 5, 20, and 80 and compare the variance of the path.

**Practice.** Change kappa in the step-size schedule and inspect convergence.

**Practice.** Use a held-out Bernoulli log score rather than one minibatch ELBO.